In [1]:
import os
import shutil

base = r"C:\Users\kogantiy\Downloads\archive\space images"

rename_map = {
    "nebula - Google Search": "nebula",
    "galaxies - Google Search": "galaxy",
    "planets - Google Search": "planet",
    "stars - Google Search": "star"
}

folders = os.listdir(base)
print("Before:", folders)

for folder in folders:
    src = os.path.join(base, folder)

    # skip invalid/junk folders
    if folder not in rename_map:
        print("Skipping:", folder)
        continue

    dst = os.path.join(base, rename_map[folder])

    if not os.path.exists(dst):
        os.rename(src, dst)
        print(f"Renamed {folder} → {rename_map[folder]}")
    else:
        print(f"Folder {rename_map[folder]} already exists — skipping rename")

print("DONE CLEANING")
print("After:", os.listdir(base))


Before: ['constellation - Google Search', 'cosmos space - Google Search', 'galaxy', 'nebula', 'planet', 'star', 'stars - Google Search - Copy', 'test', 'train']
Skipping: constellation - Google Search
Skipping: cosmos space - Google Search
Skipping: galaxy
Skipping: nebula
Skipping: planet
Skipping: star
Skipping: stars - Google Search - Copy
Skipping: test
Skipping: train
DONE CLEANING
After: ['constellation - Google Search', 'cosmos space - Google Search', 'galaxy', 'nebula', 'planet', 'star', 'stars - Google Search - Copy', 'test', 'train']


In [2]:
import os
import shutil
import random

base_dir = r"C:\Users\kogantiy\Downloads\archive\space images"
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

# create train/test directories if not exist
for d in [train_dir, test_dir]:
    if not os.path.exists(d):
        os.makedirs(d)

classes = ["nebula", "galaxy", "planet", "star"]

for cls in classes:
    src_folder = os.path.join(base_dir, cls)
    train_class_folder = os.path.join(train_dir, cls)
    test_class_folder = os.path.join(test_dir, cls)

    os.makedirs(train_class_folder, exist_ok=True)
    os.makedirs(test_class_folder, exist_ok=True)

    images = [f for f in os.listdir(src_folder) if f.lower().endswith((".jpg",".jpeg",".png",".webp",".bmp",".gif"))]

    random.shuffle(images)

    split_idx = int(len(images) * 0.8)
    train_files = images[:split_idx]
    test_files = images[split_idx:]

    for f in train_files:
        shutil.copy(os.path.join(src_folder, f), os.path.join(train_class_folder, f))

    for f in test_files:
        shutil.copy(os.path.join(src_folder, f), os.path.join(test_class_folder, f))

    print(f"{cls}: {len(train_files)} train, {len(test_files)} test")

print("Dataset successfully split into train/test folders.")


nebula: 136 train, 34 test
galaxy: 189 train, 48 test
planet: 140 train, 36 test
star: 140 train, 35 test
Dataset successfully split into train/test folders.


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models


IMG_SIZE = 224
BATCH_SIZE = 32

train_dir = r"C:\Users\kogantiy\Downloads\archive\space images\train"
test_dir  = r"C:\Users\kogantiy\Downloads\archive\space images\test"

train_ds = image_dataset_from_directory(
    train_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_ds = image_dataset_from_directory(
    test_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

# Improve performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


Found 756 files belonging to 4 classes.
Found 454 files belonging to 4 classes.
Classes: ['galaxy', 'nebula', 'planet', 'star']


In [4]:
from tensorflow.keras.applications import EfficientNetB2
base_model = EfficientNetB2(...)
from tensorflow.keras import layers, models

num_classes = len(class_names)

# Data augmentation (recommended for final project)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2)
])


# Base model with pre-trained ImageNet weights
base_model = EfficientNetB0(
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    weights="imagenet",
    pooling="avg"
)

base_model.trainable = True
# Freeze bottom 70% of layers to avoid destroying pretrained weights
for layer in base_model.layers[:200]:
    layer.trainable = False

# Build full model
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=True)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 1280)           │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,054,695 (15.47 MB)

 Trainable params: 2,055,828 (7.84 MB)

 Non-trainable params: 1,998,867 (7.63 MB)

In [5]:
import os
from PIL import Image

base = r"C:\Users\kogantiy\Downloads\archive\space images\train"

bad_files = []

for cls in os.listdir(base):
    folder = os.path.join(base, cls)
    if not os.path.isdir(folder):
        continue
    
    for file in os.listdir(folder):
        path = os.path.join(folder, file)
        try:
            img = Image.open(path)
            img.verify()  # check corruption
            img = Image.open(path).convert("RGB")  # ensure 3 channels
        except Exception as e:
            bad_files.append(path)

print("BAD FILES FOUND:", len(bad_files))
for b in bad_files:
    print(b)


BAD FILES FOUND: 1
C:\Users\kogantiy\Downloads\archive\space images\train\nebula\37.jpg


In [6]:
import os
from PIL import Image

base = r"C:\Users\kogantiy\Downloads\archive\space images"

converted = 0

for root, dirs, files in os.walk(base):
    for f in files:
        if f.lower().endswith((".jpg",".jpeg",".png",".bmp",".webp",".gif")):
            path = os.path.join(root, f)
            try:
                img = Image.open(path)
                img = img.convert("RGB")   # force RGB
                img.save(path)
                converted += 1
            except:
                pass

print("Converted images to RGB:", converted)


Converted images to RGB: 3146


In [7]:
from tensorflow.keras.callbacks import EarlyStopping

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10,
    callbacks=[early_stop]
)


Epoch 1/10
 6/24 ━━━━━━━━━━━━━━━━━━━━ 14s 801ms/step - accuracy: 0.1614 - loss: 1.5383

InvalidArgumentError: Graph execution error:

Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
jpeg::Uncompress failed. Invalid JPEG data or crop window.
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_26806]

In [ ]:

# Evaluate on test set
test_loss, test_acc = model.evaluate(test_ds)
print("Final Test Accuracy:", test_acc)

# Save model
model.save("efficientnet_model.h5")
print("Model saved as efficientnet_model.h5")

# Plot training curves
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy Curve")

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss Curve")

plt.show()


In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(test_ds)
print("Final Test Accuracy:", test_acc)

# Save model
model.save("efficientnet_model.h5")
print("Model saved as efficientnet_model.h5")

# Plot training curves
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy Curve")

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss Curve")

plt.show()
